# 05 — Recherche du meilleur modèle supervisé


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd().resolve()
while not (ROOT / "datasets").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "datasets" / "ml-research"
df = pd.read_csv(DATA / "plants_SYNTHETIC_DEMO.csv")
print("Synthetic demo — NOT horticultural evidence:", df.shape)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
# Demo target: synthetic water_need; replace with observed horticultural labels.
target="water_need"
features=["plant_type","evergreen","height_cm","spread_cm","hardiness_min_c","sun_exposure","soil_type"]
labeled=df.dropna(subset=[target]).copy()
Xraw=labeled[features]; y=labeled[target]
pre=ColumnTransformer([
    ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),["plant_type","evergreen","sun_exposure","soil_type"]),
    ("num",SimpleImputer(strategy="median"),["height_cm","spread_cm","hardiness_min_c"])
])
models={
    "dummy":DummyClassifier(strategy="most_frequent"),
    "logistic_regression":LogisticRegression(max_iter=1000),
    "decision_tree":DecisionTreeClassifier(max_depth=7,random_state=42),
    "random_forest":RandomForestClassifier(n_estimators=150,class_weight="balanced",random_state=42,n_jobs=-1),
    "hist_gradient_boosting":HistGradientBoostingClassifier(random_state=42)
}
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=42)
rows=[]
for name,estimator in models.items():
    pipeline=Pipeline([("preprocess",pre),("model",estimator)])
    scores=cross_validate(pipeline,Xraw,y,cv=cv,scoring={"f1_macro":"f1_macro","balanced_accuracy":"balanced_accuracy"})
    rows.append({"algorithm":name,"f1_macro":scores["test_f1_macro"].mean(),
                 "balanced_accuracy":scores["test_balanced_accuracy"].mean()})
scores=pd.DataFrame(rows).sort_values("f1_macro",ascending=False)
display(scores)
scores.to_csv(DATA/"supervised_scores.csv",index=False)
print("SYNTHETIC LABELS: no real-world model performance established.")
print("Group related species/cultivars when splitting real data; avoid leakage.")